# 07. Kinetic Plasma Waves & Collisionless Electrostatic Shocks
## $(k, \omega)$ Dispersion Reconstruction & Supersonic Ion Reflection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/avinash-tiwary/ePic/blob/main/notebooks/07_Kinetic_Plasma_Waves_and_Shocks.ipynb)

### 1. Theoretical Background
This advanced notebook explores two cornerstone phenomena in kinetic plasma physics:

1. **Bohm-Gross Langmuir Wave Dispersion**:
Taking the 2D spatio-temporal Fourier transform of thermal electric fluctuations $E(x, t)$ recovers the exact warm plasma dispersion surface:
$$\omega^2(k) = \omega_{pe}^2 + 3 k^2 v_{th}^2$$

2. **Collisionless Electrostatic Shocks** (Forslund & Freidberg 1971):
When two plasma slabs collide at supersonic velocities ($M = v_{\text{drift}} / c_s > 1$, where $c_s = \sqrt{T_e/m_i}$), a steep electrostatic potential barrier $\Delta \phi$ forms.
Incoming ions with kinetic energy $\frac{1}{2} m_i v_i^2 < e \Delta \phi$ are specularly reflected ahead of the shock front ($v_{\text{ref}} = 2 v_{\text{shock}} - v_{\text{in}}$), generating a precursor foot in kinetic phase space.

---

In [ ]:
# @title Setup & Environment Bootstrap
import sys
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Google Colab: Installing ePic...")
    subprocess.run(["git", "clone", "https://github.com/avinash-tiwary/ePic.git"], check=True)
    sys.path.append("ePic/src")
    subprocess.run(["pip", "install", "-q", "-e", "ePic"], check=True)
else:
    import os
    sys.path.append(os.path.abspath("../src"))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from epic.solvers.pic1d import PIC1DSolver
from epic.field.cic import deposit_charge_1d, interpolate_field_1d
from epic.field.poisson import solve_poisson_1d_fft
from epic.pusher.boris import boris_push, retard_velocity
from epic.diagnostics import apply_epic_style, format_epic_figure, EPIC_COLORS

apply_epic_style()
print("ePic unified dark astrophysics styling active!")

### 2. Experiment A: First-Principles $(k, \omega)$ Dispersion Reconstruction

In [ ]:
Nx = 256
boxsize = 20.0 * np.pi
v_th = 0.5
N_particles = 100000
dt = 0.1
Nt = 350

solver = PIC1DSolver(Nx=Nx, boxsize=boxsize, dt=dt)
weight = (1.0 * boxsize) / N_particles

np.random.seed(42)
x = np.random.uniform(0.0, boxsize, N_particles)
vel = np.random.normal(0.0, v_th, (N_particles, 3))
solver.add_species("electrons", q=-weight, m=weight, pos=x, vel=vel)
solver.initialize()

E_xt = np.zeros((Nt, Nx))
print("Recording wavefield E(x, t)...")
for t_step in range(Nt):
    E_xt[t_step, :] = solver.E.copy()
    solver.step()

# 2D FFT
window = np.hanning(Nt)[:, np.newaxis] * np.hanning(Nx)[np.newaxis, :]
fft_shift = np.fft.fftshift(np.fft.fft2((E_xt - np.mean(E_xt)) * window))
power_spec = np.abs(fft_shift) ** 2

omega_axis = np.fft.fftshift(np.fft.fftfreq(Nt, d=dt)) * 2.0 * np.pi
k_axis = np.fft.fftshift(np.fft.fftfreq(Nx, d=solver.dx)) * 2.0 * np.pi

k_mask = (k_axis >= 0.0) & (k_axis <= 1.8)
omega_mask = (omega_axis >= 0.0) & (omega_axis <= 3.2)
sub_k = k_axis[k_mask]
sub_omega = omega_axis[omega_mask]
sub_spec = power_spec[np.ix_(omega_mask, k_mask)]

# Bohm-Gross theory curve
k_th = np.linspace(0.0, 1.8, 150)
w_th = np.sqrt(1.0 + 3.0 * (k_th**2) * (v_th**2))

fig, ax = plt.subplots(figsize=(10, 6), dpi=130)
log_p = np.log10(np.maximum(sub_spec, 1e-12))
im = ax.pcolormesh(sub_k, sub_omega, log_p, cmap="turbo", shading="auto", vmin=np.percentile(log_p, 98)-4, vmax=np.percentile(log_p, 99.8))
ax.plot(k_th, w_th, color="#00ffff", lw=2.8, label=r"Bohm-Gross Theory: $\omega^2 = \omega_{pe}^2 + 3 k^2 v_{th}^2$")
ax.axhline(1.0, color="#fbbf24", linestyle=":", lw=1.8, label=r"Cold Plasma Cutoff: $\omega = \omega_{pe}$")
ax.set_xlabel(r"Wavenumber $k$ ($\omega_{pe} / c$)")
ax.set_ylabel(r"Frequency $\omega$ ($\omega_{pe}$)")
ax.set_title("(a) Reconstructed Kinetic Plasma Wave Dispersion Diagram")
ax.legend(loc="upper left")
cbar = plt.colorbar(im, ax=ax, label=r"Spectral Energy Density $\log_{10} |\tilde{E}(k, \omega)|^2$")
cbar.ax.yaxis.label.set_color(EPIC_COLORS["text"])
format_epic_figure(fig, title="First-Principles Kinetic Dispersion Relation Reconstruction")
plt.show()

### 3. Experiment B: Collisionless Electrostatic Shock Dynamics
We simulate the supersonic collision ($M = 2.0$) of two plasma streams, forming a steep electrostatic shock and reflecting ions ahead of the front.

In [ ]:
Nx = 320
boxsize = 50.0
dt = 0.05
t_end = 25.0
Nt = int(t_end / dt)
N_electrons, N_ions = 100000, 100000
m_i_ratio = 16.0
c_s = np.sqrt(1.0 / m_i_ratio)
v_drift = 2.0 * c_s

w_e = (1.0 * boxsize) / N_electrons
w_i = (1.0 * boxsize) / N_ions
q_e, m_e = -w_e, w_e
q_i, m_i = w_i, m_i_ratio * w_i

np.random.seed(42)
x_e = np.random.uniform(0.0, boxsize, N_electrons)
x_i = np.random.uniform(0.0, boxsize, N_ions)
v_e = np.random.normal(0.0, 1.0, (N_electrons, 3))
v_i = np.random.normal(0.0, np.sqrt(0.05 / m_i_ratio), (N_ions, 3))

left_e = x_e < (boxsize / 2.0)
v_e[left_e, 0] += v_drift
v_e[~left_e, 0] -= v_drift
left_i = x_i < (boxsize / 2.0)
v_i[left_i, 0] += v_drift
v_i[~left_i, 0] -= v_drift

rho = deposit_charge_1d(x_e, q_e, Nx, boxsize) + deposit_charge_1d(x_i, q_i, Nx, boxsize)
phi, E = solve_poisson_1d_fft(rho, boxsize)

E_e = interpolate_field_1d(x_e, E, Nx, boxsize).ravel()
E_i = interpolate_field_1d(x_i, E, Nx, boxsize).ravel()
E_e_vec = np.column_stack((E_e, np.zeros((N_electrons, 2))))
E_i_vec = np.column_stack((E_i, np.zeros((N_ions, 2))))
B_vec = np.zeros((N_electrons, 3))

v_e_half = retard_velocity(v_e, E_e_vec, B_vec, q_e, m_e, dt)
v_i_half = retard_velocity(v_i, E_i_vec, B_vec, q_i, m_i, dt)

print(f"Evolving supersonic collisionless shock (M={v_drift/c_s:.1f})...")
for step in range(Nt):
    E_e = interpolate_field_1d(x_e, E, Nx, boxsize).ravel()
    E_i = interpolate_field_1d(x_i, E, Nx, boxsize).ravel()
    E_e_vec[:, 0] = E_e
    E_i_vec[:, 0] = E_i
    v_e_next = boris_push(v_e_half, E_e_vec, B_vec, q_e, m_e, dt)
    v_i_next = boris_push(v_i_half, E_i_vec, B_vec, q_i, m_i, dt)
    x_e = np.mod(x_e + v_e_next[:, 0] * dt, boxsize)
    x_i = np.mod(x_i + v_i_next[:, 0] * dt, boxsize)
    rho = deposit_charge_1d(x_e, q_e, Nx, boxsize) + deposit_charge_1d(x_i, q_i, Nx, boxsize)
    phi, E = solve_poisson_1d_fft(rho, boxsize)
    v_e_half = v_e_next
    v_i_half = v_i_next

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), dpi=130)
sub_i = slice(None, None, 5)
ax1.scatter(x_i[sub_i], v_i_half[sub_i, 0] / c_s, s=1.0, color=EPIC_COLORS["cyan"], alpha=0.5, label="Ions")
ax1.axvline(boxsize/2.0, color=EPIC_COLORS["gold"], linestyle="--", lw=1.8, label="Shock Collision Center")
ax1.set_xlim(0, boxsize)
ax1.set_ylim(-3.5, 3.5)
ax1.set_xlabel(r"Position $x$ ($c/\omega_{pe}$)")
ax1.set_ylabel(r"Ion Velocity $v_{i, x} / c_s$")
ax1.set_title("(a) Kinetic Ion Phase Space & Reflected Shock Foot")
ax1.legend(loc="upper right")

grid_x = np.linspace(0.0, boxsize, Nx)
ax2.plot(grid_x, phi, color=EPIC_COLORS["emerald"], lw=2.2, label=r"Potential $\phi(x)$")
ax2_tw = ax2.twinx()
ax2_tw.plot(grid_x, E, color=EPIC_COLORS["gold"], lw=1.8, linestyle="-.", label=r"Electric Field $E(x)$")
ax2.set_xlim(0, boxsize)
ax2.set_xlabel(r"Position $x$ ($c/\omega_{pe}$)")
ax2.set_ylabel(r"Potential $\phi$", color=EPIC_COLORS["emerald"])
ax2_tw.set_ylabel(r"Shock Field $E$", color=EPIC_COLORS["gold"])
ax2.set_title("(b) Self-Consistent Shock Potential Jump & Ramp")

format_epic_figure(fig, title="ePic 1D-3V Collisionless Electrostatic Shock Dynamics", subtitle="Supersonic Mach 2 Collision, Specular Reflection, and Barrier Formation")
plt.show()